In [1]:
import mudata
import logging as log
from scenicplus.networks import create_nx_tables, create_nx_graph
from pycisTopic.diff_features import find_highly_variable_features

from scenicplus.scenicplus_class import mudata_to_scenicplus
from scenicplus.preprocessing.filtering import apply_std_filtering_to_eRegulons
import pandas as pd

In [2]:
import networkx as nx

In [4]:
scplus_mdata = mudata.read("../process/20241226_scenicplus_outs/scplusmdata.h5mu")

/home/zhanglab/micromamba/envs/py311/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/zhanglab/micromamba/envs/py311/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/zhanglab/micromamba/envs/py311/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/zhanglab/micromamba/envs/py311/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/zhanglab/micromamba/envs/py311/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/zhanglab/micromamba

In [8]:
hvg = pd.read_csv("../../2024.4_scATAC/processed_data/variable_gene/5.1_rna_var_scanpy_2000.csv",index_col=0)["0"]

In [10]:
hvr = find_highly_variable_features(
    pd.DataFrame(scplus_mdata["scATAC_counts"].X).T,
    n_top_features=2000,
    plot = False
)

2025-02-15 10:57:45,430 cisTopic     INFO     Calculating mean
2025-02-15 10:58:41,651 cisTopic     INFO     Calculating variance


/home/zhanglab/micromamba/envs/py311/lib/python3.11/site-packages/pycisTopic/diff_features.py:649: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


2025-02-15 11:07:49,353 cisTopic     INFO     Done!


<Figure size 640x480 with 0 Axes>

In [11]:
hvr = scplus_mdata["scATAC_counts"].var_names[hvr]

In [16]:
hvr2 = find_highly_variable_features(
    pd.DataFrame(scplus_mdata["scATAC_counts"].X).T,
    n_top_features=1000,
    plot = False
)

2025-02-15 11:35:58,914 cisTopic     INFO     Calculating mean
2025-02-15 11:37:03,659 cisTopic     INFO     Calculating variance


/home/zhanglab/micromamba/envs/py311/lib/python3.11/site-packages/pycisTopic/diff_features.py:649: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


2025-02-15 11:47:23,940 cisTopic     INFO     Done!


<Figure size 640x480 with 0 Axes>

In [17]:
hvr2 = scplus_mdata["scATAC_counts"].var_names[hvr2]

In [12]:
nx_tables = create_nx_tables(
        scplus_obj = scplus_mdata,
        eRegulon_metadata_key ='extended_e_regulon_metadata',
        subset_regions = hvr,
        subset_genes = hvg
    )

In [18]:
hvg2 = pd.read_csv("../process/SNP/SNP2ell/20250215_vargene1000_1.csv",index_col=0)

In [20]:
hvg2["0"]

0      1700034P13Rik
1              Sulf1
2            Gm29570
3                Msc
4               Jph1
           ...      
995            Eif3a
996           mt-Co3
997           mt-Nd3
998              GFP
999             Il24
Name: 0, Length: 1000, dtype: object

In [21]:
nx_tables2 = create_nx_tables(
        scplus_obj = scplus_mdata,
        eRegulon_metadata_key ='extended_e_regulon_metadata',
        subset_regions = hvr2,
        subset_genes = hvg2["0"]
    )

In [91]:
nx_tables2_full = create_nx_tables(
        scplus_obj = scplus_mdata,
        eRegulon_metadata_key ='regulon_metadata',
        subset_regions = hvr2,
        subset_genes = hvg2["0"]
    )

In [94]:
G2_full, pos, edge_tables, node_tables = create_nx_graph(
        nx_tables2_full,
        use_edge_tables = ['TF2R','R2G'],
        shape_node_by = {
            'TF': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
            'Gene': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
            'Region': {'variable': 'fixed_shape', 'fixed_shape': 'diamond'}
        },
        size_node_by = {
            'TF': {'variable': 'fixed_size', 'fixed_size': 30},
            'Gene': {'variable': 'fixed_size', 'fixed_size': 15},
            'Region': {'variable': 'fixed_size', 'fixed_size': 10}
        },
        label_size_by = {
            'TF': {'variable': 'fixed_label_size', 'fixed_label_size': 18.0},
            'Gene': {'variable': 'fixed_label_size', 'fixed_label_size': 12.0},
            'Region': {'variable': 'fixed_label_size', 'fixed_label_size': 6.0}
        },
        width_edge_by = {'R2G': {'variable' : 'importance_R2G', 'max_size' :  1.5, 'min_size' : 1}},
        transparency_edge_by =  {'R2G': {'variable' : 'importance_R2G', 'min_alpha': 0.1, 'v_min': 0}},
        layout = 'kamada_kawai_layout',
        scale_position_by = 250,
    )

In [96]:
nx.to_pandas_edgelist(G2_full, source="TF", target="gene").to_csv("../process/SNP/SNP2ell/20250226_network_table_1000_full.csv")

In [14]:
G, pos, edge_tables, node_tables = create_nx_graph(
        nx_tables,
        use_edge_tables = ['TF2R','R2G'],
        shape_node_by = {
            'TF': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
            'Gene': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
            'Region': {'variable': 'fixed_shape', 'fixed_shape': 'diamond'}
        },
        size_node_by = {
            'TF': {'variable': 'fixed_size', 'fixed_size': 30},
            'Gene': {'variable': 'fixed_size', 'fixed_size': 15},
            'Region': {'variable': 'fixed_size', 'fixed_size': 10}
        },
        label_size_by = {
            'TF': {'variable': 'fixed_label_size', 'fixed_label_size': 18.0},
            'Gene': {'variable': 'fixed_label_size', 'fixed_label_size': 12.0},
            'Region': {'variable': 'fixed_label_size', 'fixed_label_size': 6.0}
        },
        width_edge_by = {'R2G': {'variable' : 'importance_R2G', 'max_size' :  1.5, 'min_size' : 1}},
        transparency_edge_by =  {'R2G': {'variable' : 'importance_R2G', 'min_alpha': 0.1, 'v_min': 0}},
        layout = 'kamada_kawai_layout',
        scale_position_by = 250,
    )

In [22]:
G2, pos, edge_tables, node_tables = create_nx_graph(
        nx_tables2,
        use_edge_tables = ['TF2R','R2G'],
        shape_node_by = {
            'TF': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
            'Gene': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
            'Region': {'variable': 'fixed_shape', 'fixed_shape': 'diamond'}
        },
        size_node_by = {
            'TF': {'variable': 'fixed_size', 'fixed_size': 30},
            'Gene': {'variable': 'fixed_size', 'fixed_size': 15},
            'Region': {'variable': 'fixed_size', 'fixed_size': 10}
        },
        label_size_by = {
            'TF': {'variable': 'fixed_label_size', 'fixed_label_size': 18.0},
            'Gene': {'variable': 'fixed_label_size', 'fixed_label_size': 12.0},
            'Region': {'variable': 'fixed_label_size', 'fixed_label_size': 6.0}
        },
        width_edge_by = {'R2G': {'variable' : 'importance_R2G', 'max_size' :  1.5, 'min_size' : 1}},
        transparency_edge_by =  {'R2G': {'variable' : 'importance_R2G', 'min_alpha': 0.1, 'v_min': 0}},
        layout = 'kamada_kawai_layout',
        scale_position_by = 250,
    )

In [15]:
nx.to_pandas_edgelist(G, source="TF", target="gene").to_csv("../process/SNP/SNP2ell/20250215_network_table_2000.csv")

In [23]:
nx.to_pandas_edgelist(G2, source="TF", target="gene").to_csv("../process/SNP/SNP2ell/20250215_network_table_1000.csv")

In [27]:
scplus_mdata

MuData object with n_obs × n_vars = 41586 × 520615
  uns:	'direct_e_regulon_metadata', 'extended_e_regulon_metadata'
  6 modalities
    scRNA_counts:	41586 x 23143
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'group', 'SCT_snn_res.0.6', 'celltype', 'SCT_snn_res.0.4', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'ident', 'knn_label'
      obsm:	'HARMONY', 'PCA', 'UMAP'
    scATAC_counts:	41586 x 496496
      obs:	'fraction_of_fragments_in_peaks', 'unique_fragments_in_peaks_count', 'pdf_values_for_duplication_ratio', 'duplication_count', 'pdf_values_for_fraction_of_fragments_in_peaks', 'sample_id', 'cisTopic_nr_acc', 'barcode_rank', 'log10_total_fragments_in_peaks_count', 'log10_total_fragments_count', 'cisTopic_log_nr_frag', 'total_fragments_in_peaks_count', 'total_fragments_count', 'duplication_ratio', 'cisTopic_log_nr_acc', 'log10_unique_fragments_count', 'log10_unique_fragments_in_peaks_count', 'cisTopic_nr_frag', 'unique_fragments_count', 'pdf_values_for_tss_enrichment', 'tss_enrichment', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'nucleosome_percentile', 'nucleosome_group', 'TSS.enrichment', 'TSS.percentile', 'high.tss', 'total', 'duplicate', 'chimeric', 'unmapped', 'lowmapq', 'mitochondrial', 'nonprimary', 'passed_filters', 'is__cell_barcode', 'excluded_reason', 'TSS_fragments', 'DNase_sensitive_region_fragments', 'enhancer_region_fragments', 'promoter_region_fragments', 'on_target_fragments', 'blacklist_region_fragments', 'peak_region_fragments', 'peak_region_cutsites', 'pct_reads_in_peaks', 'blacklist_ratio', 'n_fragment', 'frac_dup', 'scDblFinderScore', 'peaks_snn_res.0.8', 'seurat_clusters', 'nCount_RNA', 'nFeature_RNA', 'predicted.id', 'prediction.score.pe4', 'prediction.score.pe2', 'prediction.score.pe3', 'prediction.score.pe1', 'prediction.score.pe6', 'prediction.score.pe11', 'prediction.score.pe5', 'prediction.score.pe10', 'prediction.score.pe8', 'prediction.score.pe13', 'prediction.score.pe7', 'prediction.score.pe12', 'prediction.score.pe9', 'prediction.score.max', 'archR_label', 'coarseLabel', 'batch', 'FRIP', 'logCount', 'group', 'coarse_label', 'rna_cluster', 'previous_cluster', 'peakvi_cluster', 'combined_cluster', 'knn_label', 'Sample', 'barcodes', 'barcode', 'pycisTopic_leiden_10_0.6', 'pycisTopic_leiden_10_1.2', 'pycisTopic_leiden_10_3'
      var:	'Chromosome', 'Start', 'End', 'Width', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc'
      obsm:	'UMAP'
    direct_gene_based_AUC:	41586 x 173
    direct_region_based_AUC:	41586 x 173
    extended_gene_based_AUC:	41586 x 315
    extended_region_based_AUC:	41586 x 315

In [28]:
scplus_mdata.uns["regulon_metadata"] = scplus_mdata.uns["direct_e_regulon_metadata"].copy()

In [44]:
scplus_mdata.uns["regulon_metadata"] = pd.concat([scplus_mdata.uns["direct_e_regulon_metadata"], 
                                                  scplus_mdata.uns["extended_e_regulon_metadata"]],axis=0)

In [41]:
set(scplus_mdata.uns["regulon_metadata"]["eRegulon_name"])

{'Acaa1b_extended_+/+',
 'Ahr_extended_+/-',
 'Arid3a_extended_+/+',
 'Arid5b_extended_+/+',
 'Arid5b_extended_+/-',
 'Atf1_direct_+/-',
 'Atf1_extended_+/-',
 'Atf3_direct_+/+',
 'Atf3_direct_-/-',
 'Atf3_extended_+/+',
 'Atf3_extended_-/-',
 'Atf4_direct_+/-',
 'Atf4_extended_+/-',
 'Atf6_direct_+/-',
 'Atf7_direct_+/+',
 'Atf7_extended_+/+',
 'Bach1_direct_+/+',
 'Bach1_extended_+/+',
 'Barx1_extended_+/+',
 'Barx1_extended_+/-',
 'Barx1_extended_-/+',
 'Barx2_extended_+/-',
 'Bcl11a_extended_+/+',
 'Bcl11a_extended_+/-',
 'Bhlhe40_extended_+/+',
 'Bhlhe40_extended_+/-',
 'Bnc1_extended_+/+',
 'Bnc2_extended_+/+',
 'Bnc2_extended_+/-',
 'Brca1_extended_+/+',
 'Cebpa_direct_+/+',
 'Cebpb_direct_+/+',
 'Cebpb_extended_+/+',
 'Cebpb_extended_+/-',
 'Cebpg_direct_+/+',
 'Cebpg_extended_+/+',
 'Cebpg_extended_+/-',
 'Cebpz_extended_+/+',
 'Cenpb_extended_+/+',
 'Cenpb_extended_+/-',
 'Chd1_extended_+/-',
 'Chd2_direct_+/-',
 'Chd2_extended_+/-',
 'Churc1_extended_+/+',
 'Creb1_direct_+/-

In [59]:
import numpy as np
Grhl1_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Grhl1"]["TF"]))
Grhl3_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Grhl3"]["TF"]))
Arid3a_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Arid3a"]["TF"]))

In [63]:
Ahr_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Ahr"]["TF"]))
Klf7_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Klf7"]["TF"]))
Zbtb7a_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Zbtb7a"]["TF"]))
Tfap2c_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Tfap2c"]["TF"]))
Cebpb_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Cebpb"]["TF"]))
Klf5_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Klf5"]["TF"]))

In [66]:
from itertools import combinations
from collections import defaultdict

def upset_intersections(lists, list_names=None):
    """
    Analyze all possible intersections between lists, similar to UpSetR.
    
    Args:
        lists: List of lists to analyze
        list_names: Optional names for the lists (defaults to List_1, List_2, etc.)
        
    Returns:
        Dictionary with intersection information
    """
    if list_names is None:
        list_names = [f"List_{i+1}" for i in range(len(lists))]
    
    if len(lists) != len(list_names):
        raise ValueError("Number of lists and list names must match")
    
    # Convert lists to sets for efficient intersection
    sets = [set(lst) for lst in lists]
    
    # Create a dictionary to store all elements and which sets they belong to
    element_membership = defaultdict(set)
    for i, s in enumerate(sets):
        for element in s:
            element_membership[element].add(i)
    
    # Generate all possible combinations of sets
    results = {}
    for r in range(1, len(sets) + 1):
        for combo_indices in combinations(range(len(sets)), r):
            # Get the sets in this combination
            combo_names = [list_names[i] for i in combo_indices]
            combo_key = " ∩ ".join(combo_names)
            
            # Find elements that are in exactly these sets and no others
            exact_elements = [
                element for element, memberships in element_membership.items()
                if set(combo_indices) == memberships
            ]
            
            # Find elements that are in at least these sets (may be in others too)
            at_least_elements = [
                element for element, memberships in element_membership.items()
                if set(combo_indices).issubset(memberships)
            ]
            
            results[combo_key] = {
                "exact_match_elements": exact_elements,
                "exact_match_count": len(exact_elements),
                "at_least_elements": at_least_elements,
                "at_least_count": len(at_least_elements),
                "sets_involved": combo_names
            }
    
    return results



In [72]:
intersections = upset_intersections([Ahr_regulator, Klf7_regulator, Zbtb7a_regulator,Tfap2c_regulator,Cebpb_regulator,Klf5_regulator], 
           ["Ahr","Klf7","Zbtb7a","Tfap2c","Cebpb","Klf5"])

In [73]:

# Print results in a readable format
for intersection, data in intersections.items():
    print(f"\n{intersection}:")
    print(f"  Exact elements (only in these sets): {data['exact_match_elements']} (Count: {data['exact_match_count']})")
    print(f"  At least elements (in these and possibly other sets): {data['at_least_elements']} (Count: {data['at_least_count']})")


Ahr:
  Exact elements (only in these sets): ['Srebf2', 'Ahr', 'Rela', 'Hivep1'] (Count: 4)
  At least elements (in these and possibly other sets): ['Srebf2', 'Sox11', 'Ahr', 'Cebpb', 'Klf7', 'Rela', 'Grhl1', 'Fosb', 'Fosl2', 'Klf6', 'Hivep1'] (Count: 11)

Klf7:
  Exact elements (only in these sets): ['Cux1', 'Maf', 'Creb5', 'Irf6', 'Foxq1'] (Count: 5)
  At least elements (in these and possibly other sets): ['Sox11', 'Cebpb', 'Klf7', 'Fosl2', 'Klf6', 'Jund', 'Jun', 'Cux1', 'Maf', 'Creb5', 'Irf6', 'Grhl3', 'Foxq1', 'Klf4', 'Atf3', 'Tgif1'] (Count: 16)

Zbtb7a:
  Exact elements (only in these sets): ['Junb', 'Klf5', 'Klf13', 'Zbtb7a'] (Count: 4)
  At least elements (in these and possibly other sets): ['Cebpb', 'Grhl1', 'Klf6', 'Jund', 'Jun', 'Klf4', 'Atf3', 'Junb', 'Klf3', 'Klf5', 'Klf13', 'Zbtb7a'] (Count: 12)

Tfap2c:
  Exact elements (only in these sets): [] (Count: 0)
  At least elements (in these and possibly other sets): [] (Count: 0)

Cebpb:
  Exact elements (only in these sets): 

In [60]:
np.intersect1d(Grhl3_regulator,Grhl1_regulator)

array(['Cebpb', 'Grhl1', 'Klf5', 'Tfap2c'], dtype='<U6')

In [61]:
np.intersect1d(Arid3a_regulator,Grhl1_regulator)

array(['Ahr', 'Zbtb7a'], dtype='<U6')

In [78]:
Irf6_regulator = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Irf6"]["TF"]))

In [81]:
Irf6_regulator

[]

In [80]:
np.intersect1d(A,Irf6_regulator)

array([], dtype='<U32')

In [62]:
np.intersect1d(Arid3a_regulator,Grhl3_regulator)

array(['Klf7'], dtype='<U6')

In [ ]:
Ahr,Klf7,Zbtb7a,Tfap2c,Cebpb,Klf5

In [57]:
Grhl1_regulator

array({'Junb', 'Ahr', 'Cebpb', 'Vps4b', 'Fos', 'Klf3', 'Plagl1', 'Grhl1', 'Klf4', 'Mafb', 'Tfap2c', 'Klf5', 'Klf6', 'Zbtb7a'},
      dtype=object)

In [58]:
Arid3a_regulator

array({'Hif1a', 'Maf', 'Ahr', 'Elf1', 'Klf7', 'Arid3a', 'Zbtb7a'},
      dtype=object)

In [ ]:

Arid3a_regulator = np.array(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["Gene"] == "Grhl1"]["TF"]))

In [39]:
hvg

0                Xkr4
1               Sox17
2               Sntg1
3       1700034P13Rik
4                Cpa6
            ...      
1995        Serpina3n
1996             Tff1
1997          Sult1c1
1998          Olfr206
1999           Arid3a
Name: 0, Length: 2000, dtype: object

In [ ]:
nx_tables_ = create_nx_tables(
        scplus_obj = scplus_mdata,
        eRegulon_metadata_key ='regulon_metadata',
        subset_regions = hvr,
        subset_genes = hvg,subset_eRegulons=["Arid3a_extended_+/+",'Grhl1_extended_+/+',""]
    )

In [84]:
Grhl1_gene = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["TF"] == "Grhl1"]["Gene"]))
Grhl3_gene = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["TF"] == "Grhl3"]["Gene"]))
Arid3a_gene = list(set(scplus_mdata.uns["regulon_metadata"][scplus_mdata.uns["regulon_metadata"]["TF"] == "Arid3a"]["Gene"]))

In [86]:
intersections_gene = upset_intersections([Grhl1_gene, Grhl3_gene, Arid3a_gene], 
           ["Grhl1","Grhl3","Arid3a"])

In [87]:
intersections_gene

{'Grhl1': {'exact_match_elements': ['Kcnj15',
   'Cpm',
   'Hdhd3',
   'Alcam',
   'Pik3r1',
   'Map6',
   'Gsdmc',
   'Hbegf',
   'Erbb3',
   'Cds1',
   'Lrrfip1',
   'Ifi27',
   'Alox12',
   'Sprr2k',
   'Ankrd22',
   'Krt1',
   'Paqr5',
   'Taok3',
   'Irf2',
   'Gclc',
   'Vsig10',
   'Cers3',
   'Vat1',
   'Pkp2',
   'Lgals3',
   'Cldn23',
   'Mgat4a',
   'Stfa3',
   'Capn5',
   'Plekhh3',
   'Krt18',
   'Stbd1',
   'Ttc9',
   'Glrx',
   'Emp2',
   'Erbb2',
   'Rapgefl1',
   'Ppp1r3b',
   'Rit1',
   'Krt8',
   'Ahr',
   'Tacc2',
   'Exph5',
   'Parp4',
   'Plin2',
   'Irak2',
   'Mafb',
   'Macc1',
   'Muc4',
   'Gbp2',
   'Scrn1',
   'Ankra2',
   'Gpr153',
   'Zbtb7a',
   'Gpr157',
   'Map3k1',
   'Ovol1',
   'Capns2',
   'Rassf8',
   'Idi2',
   'Ahnak2',
   'Map3k6',
   'Cdc42ep1',
   'Grhl1',
   'Calml3',
   'Fam174b',
   'Clic5',
   'Tmprss3',
   'Pdzk1ip1',
   'Krt77',
   'Eif2ak2',
   'Ezr',
   'Casp14',
   'Dnaja4',
   'Rhbg',
   'Pla2g4a',
   'Cast',
   'Krt79',
   'Gm266'